# FraudShield Advanced: Production Monitoring Lab

## Scenario

Your FraudShield model is deployed to production and processing live
e-commerce transactions. The data science team needs production monitoring
to detect when incoming data drifts from the distributions the model was
trained on. Without monitoring, model degradation goes unnoticed until
business metrics -- fraud losses, false positive rates -- visibly worsen,
which can take weeks or months.

In this lab, you will:

1. Deploy a FraudShield endpoint with data capture enabled
2. Create a data quality baseline from training data
3. Configure a monitoring schedule that runs automatically
4. Simulate data drift by sending shifted feature distributions
5. Detect and analyze the resulting violations
6. Clean up all resources

---

*This notebook runs on ml.t3.medium in SageMaker Studio. Endpoints and
monitoring jobs use ml.m5.xlarge (Free Tier eligible).*

In [ ]:
%pip install -q sagemaker boto3 pandas numpy

import sagemaker
import boto3
import json
import time
import numpy as np
import pandas as pd
from datetime import datetime
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    DataCaptureConfig,
    CronExpressionGenerator,
)
from sagemaker.model_monitor.dataset_format import DatasetFormat
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer
from sagemaker.xgboost import XGBoostModel

sm_session = sagemaker.Session()
sm_client = sm_session.sagemaker_client
s3_client = boto3.client("s3")
region = sm_session.boto_region_name
role = sagemaker.get_execution_role()
default_bucket = sm_session.default_bucket()

PREFIX = "fraudshield-monitoring-lab"
ENDPOINT_NAME = f"fraudshield-lab-ep-{datetime.now().strftime('%Y%m%d-%H%M')}"
CAPTURE_S3 = f"s3://{default_bucket}/{PREFIX}/data-capture"
BASELINE_S3 = f"s3://{default_bucket}/{PREFIX}/baseline"
REPORTS_S3 = f"s3://{default_bucket}/{PREFIX}/reports"
TRAIN_S3 = f"s3://{default_bucket}/{PREFIX}/processed/train.csv"
MODEL_S3 = f"s3://{default_bucket}/{PREFIX}/model/model.tar.gz"

print(f"Region         : {region}")
print(f"Role           : {role.split('/')[-1]}")
print(f"Default bucket : {default_bucket}")
print(f"Endpoint name  : {ENDPOINT_NAME}")

## Concept Connection: Why Monitor?

Two types of production drift can degrade a deployed model:

**Data drift** -- the statistical distribution of input features changes
compared to training data. For example, if transaction amounts in production
shift from a mean of $150 to $800, the model is receiving inputs it was not
trained to handle. Data drift does not always cause model quality degradation
(the drifted feature may have low importance), but it is a leading indicator.

**Concept drift** -- the relationship between features and the target variable
changes. The same transaction profile that indicated fraud six months ago may
now be legitimate due to changing fraud patterns. Concept drift causes model
quality degradation even when the input distribution is stable.

Data Quality Monitoring detects data drift by comparing live inference data
against the baseline statistics and constraints. Model Quality Monitoring
detects concept drift by comparing predictions against ground truth labels.
Together, they provide a complete picture of production model health.

## Step 1: Deploy with Data Capture

Deploy the FraudShield endpoint with `DataCaptureConfig` enabled. This
captures 100% of inference requests and responses to S3, which the
monitoring schedule will analyze.

In [ ]:
data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=CAPTURE_S3,
    capture_options=["Input", "Output"],
    csv_content_types=["text/csv"],
)

xgb_model = XGBoostModel(
    model_data=MODEL_S3,
    role=role,
    framework_version="1.5-1",
    sagemaker_session=sm_session,
)

predictor = xgb_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name=ENDPOINT_NAME,
    data_capture_config=data_capture_config,
)

predictor.serializer = CSVSerializer()
predictor.deserializer = CSVDeserializer()

print(f"Endpoint deployed: {ENDPOINT_NAME}")

In [ ]:
capture_prefix = f"{PREFIX}/data-capture/{ENDPOINT_NAME}"
response = s3_client.list_objects_v2(
    Bucket=default_bucket, Prefix=capture_prefix, MaxKeys=10
)

if "Contents" in response:
    print(f"Data capture verified: {len(response['Contents'])} file(s) in S3.")
    for obj in response["Contents"][:3]:
        print(f"  {obj['Key']}  ({obj['Size']} bytes)")
else:
    print("No capture files found yet. Wait and re-run this cell.")

In [ ]:
np.random.seed(42)
num_invocations = 50

for i in range(num_invocations):
    amount = np.random.uniform(10, 500)
    hour = np.random.randint(0, 24)
    category = np.random.randint(0, 5)
    age = np.random.randint(18, 75)
    distance = np.random.uniform(0, 100)

    payload = f"{amount:.2f},{hour},{category},{age},{distance:.2f}"
    response = predictor.predict(payload)

print(f"Sent {num_invocations} normal invocations to {ENDPOINT_NAME}")
print("Waiting 60 seconds for data capture files to appear in S3...")
time.sleep(60)

## Step 2: Create a Baseline

Run `suggest_baseline()` on the training data to produce `statistics.json`
and `constraints.json`. These files define what "normal" looks like for
each feature.

In [ ]:
my_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
)

my_monitor.suggest_baseline(
    baseline_dataset=TRAIN_S3,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=BASELINE_S3,
    wait=True,
    logs=False,
)

print("Baseline job complete.")
print(f"Baseline artifacts: {BASELINE_S3}")
print()

stats_key = f"{PREFIX}/baseline/statistics.json"
constraints_key = f"{PREFIX}/baseline/constraints.json"

stats_obj = s3_client.get_object(Bucket=default_bucket, Key=stats_key)
stats = json.loads(stats_obj["Body"].read().decode("utf-8"))

constraints_obj = s3_client.get_object(Bucket=default_bucket, Key=constraints_key)
constraints = json.loads(constraints_obj["Body"].read().decode("utf-8"))

print("=== Baseline Statistics ===")
for feature in stats.get("features", [])[:5]:
    name = feature.get("name", "N/A")
    numerical = feature.get("numerical_statistics", {})
    if numerical:
        print(f"  {name}: mean={numerical.get('mean', 'N/A'):.4f}, "
              f"std={numerical.get('std_dev', 'N/A'):.4f}, "
              f"min={numerical.get('min', 'N/A')}, max={numerical.get('max', 'N/A')}")

print()
print("=== Baseline Constraints ===")
for feature in constraints.get("features", [])[:5]:
    name = feature.get("name", "N/A")
    ftype = feature.get("inferred_type", "N/A")
    print(f"  {name}: type={ftype}, completeness={feature.get('completeness', 'N/A')}")

### Presentation Checkpoint

Before proceeding, be prepared to explain:

1. **What does `statistics.json` contain?** Per-feature statistics computed
   from the training data: mean, standard deviation, min, max, unique value
   counts, missing value percentages, and distribution histograms.

2. **What does `constraints.json` contain?** Rules derived from the statistics:
   expected data types, nullability requirements, allowed value sets for
   categorical features, and acceptable numeric ranges.

3. **Why are these files the foundation of monitoring?** Every monitoring
   execution compares live data against these files. Without them, there is
   no reference to measure drift against.

## Step 3: Configure Monitoring Schedule

Create an hourly monitoring schedule that compares captured inference data
against the baseline.

In [ ]:
SCHEDULE_NAME = f"fraudshield-lab-schedule-{datetime.now().strftime('%Y%m%d-%H%M')}"

my_monitor.create_monitoring_schedule(
    monitor_schedule_name=SCHEDULE_NAME,
    endpoint_input=ENDPOINT_NAME,
    output_s3_uri=REPORTS_S3,
    statistics=my_monitor.baseline_statistics(),
    constraints=my_monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

print(f"Monitoring schedule created: {SCHEDULE_NAME}")

In [ ]:
schedule_desc = sm_client.describe_monitoring_schedule(
    MonitoringScheduleName=SCHEDULE_NAME
)

print(f"Schedule Name   : {schedule_desc['MonitoringScheduleName']}")
print(f"Schedule Status : {schedule_desc['MonitoringScheduleStatus']}")
print(f"Endpoint        : {schedule_desc['EndpointName']}")
print(f"Creation Time   : {schedule_desc['CreationTime']}")

schedule_config = schedule_desc["MonitoringScheduleConfig"]
print(f"Cron Expression : {schedule_config['ScheduleConfig']['ScheduleExpression']}")

if schedule_desc["MonitoringScheduleStatus"] == "Scheduled":
    print("\nSchedule is active and will execute at the next hourly window.")
else:
    print(f"\nSchedule status is {schedule_desc['MonitoringScheduleStatus']}.")

### Concept Connection: Statistical Tests Behind Drift Detection

Model Monitor uses different statistical tests depending on feature type:

- **Continuous features (numeric):** Kolmogorov-Smirnov (K-S) test. The K-S
  statistic (D) measures the maximum difference between the cumulative
  distribution functions of baseline and live data. D near 0 means no drift;
  D near 1 means complete distribution change.

- **Categorical features:** Chi-Squared test. Compares the frequency of each
  category in baseline vs. live data. A large Chi-Squared value means the
  category proportions have shifted.

When you examine violation reports below, note which test was applied to each
flagged feature and whether the effect size (not just the p-value) indicates
operationally meaningful drift.

## Step 4: Simulate Drift

Send invocations with deliberately shifted feature distributions to trigger
violations. The `transaction_amount` feature is shifted from its normal range
($10-$500) to a much higher range ($5,000-$50,000), and the `age` feature
is shifted outside the trained range.

In [ ]:
print("Checking monitoring execution history...")
print()

executions = sm_client.list_monitoring_executions(
    MonitoringScheduleName=SCHEDULE_NAME,
    MaxResults=5,
)["MonitoringExecutionSummaries"]

if executions:
    for ex in executions:
        print(f"Execution Time : {ex['ScheduledTime']}")
        print(f"Status         : {ex['MonitoringExecutionStatus']}")
        failure = ex.get("FailureReason", "")
        if failure:
            print(f"Failure Reason : {failure}")
        print()
else:
    print("No executions recorded yet.")
    print("The schedule must complete at least one hourly cycle.")

In [ ]:
np.random.seed(99)
num_drifted = 100

print("Sending drifted invocations...")
for i in range(num_drifted):
    amount = np.random.uniform(5000, 50000)
    hour = np.random.randint(0, 24)
    category = np.random.randint(0, 8)
    age = np.random.randint(80, 120)
    distance = np.random.uniform(200, 1000)

    payload = f"{amount:.2f},{hour},{category},{age},{distance:.2f}"
    response = predictor.predict(payload)

print(f"Sent {num_drifted} drifted invocations.")
print(f"  transaction_amount shifted to $5,000-$50,000 (baseline: $10-$500)")
print(f"  age shifted to 80-120 (baseline: 18-75)")
print(f"  distance shifted to 200-1000 (baseline: 0-100)")
print(f"  category expanded to 0-7 (baseline: 0-4)")
print()
print("The next monitoring execution will compare these invocations")
print("against the baseline and should detect violations.")

In [ ]:
print("Checking for violation reports...")
print("(The monitoring schedule must execute at least once after the")
print(" drifted invocations were sent.)")
print()

response = s3_client.list_objects_v2(
    Bucket=default_bucket, Prefix=f"{PREFIX}/reports"
)

violation_files = []
if "Contents" in response:
    violation_files = [
        obj["Key"]
        for obj in response["Contents"]
        if "constraint_violations" in obj["Key"]
    ]

if violation_files:
    latest_key = sorted(violation_files)[-1]
    print(f"Found {len(violation_files)} violation report(s).")
    print(f"Latest: {latest_key}")
    print()

    v_obj = s3_client.get_object(Bucket=default_bucket, Key=latest_key)
    violations = json.loads(v_obj["Body"].read().decode("utf-8"))

    print(f"Total violations: {len(violations.get('violations', []))}")
    print()
    for v in violations.get("violations", []):
        print(f"Feature    : {v.get('feature_name', 'N/A')}")
        print(f"Type       : {v.get('constraint_check_type', 'N/A')}")
        print(f"Description: {v.get('description', 'N/A')}")
        print()
else:
    print("No violation reports found yet.")
    print("Wait for the next scheduled monitoring execution, then re-run this cell.")

### Presentation Checkpoint

Be prepared to present:

1. **Show the violation report** -- which features were flagged, and what
   violation types were raised?

2. **Identify drifted features** -- which features drifted and by how much?
   Which statistical test was used (K-S for continuous features, Chi-Squared
   for categorical features)?

3. **Operational response** -- if this were a real production violation, what
   would the next steps be? (Investigate root cause, check if model quality
   has degraded, consider retraining on recent data)

---

## MANDATORY Cleanup

Delete all resources to avoid ongoing charges. Always delete monitoring
schedules before the endpoint -- a schedule referencing a deleted endpoint
will produce errors in CloudWatch.

In [ ]:
print("=== Cleanup ===")
print()

print("1. Deleting monitoring schedule...")
try:
    my_monitor.delete_monitoring_schedule()
    print(f"   Deleted: {SCHEDULE_NAME}")
except Exception as e:
    print(f"   Schedule deletion: {e}")

print()
print("2. Deleting endpoint...")
try:
    predictor.delete_endpoint(delete_endpoint_config=True)
    print(f"   Deleted: {ENDPOINT_NAME}")
except Exception as e:
    print(f"   Endpoint deletion: {e}")

print()
print("3. Deleting model...")
try:
    predictor.delete_model()
    print("   Deleted model.")
except Exception as e:
    print(f"   Model deletion: {e}")

print()
print("=== Verifying cleanup ===")
try:
    sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    print(f"WARNING: Endpoint {ENDPOINT_NAME} still exists.")
except sm_client.exceptions.ClientError:
    print(f"Endpoint {ENDPOINT_NAME} confirmed deleted.")

schedules = sm_client.list_monitoring_schedules(
    EndpointName=ENDPOINT_NAME
)["MonitoringScheduleSummaries"]
if schedules:
    print(f"WARNING: {len(schedules)} schedule(s) still active.")
else:
    print("All monitoring schedules confirmed deleted.")

print()
print("Cleanup complete. Verify in the SageMaker console that no")
print("endpoints or monitoring schedules remain active.")

### Lab Summary

In this lab you completed the full production monitoring workflow:

1. Deployed an endpoint with data capture sampling 100% of inference traffic
2. Created a data quality baseline that defines expected feature distributions
3. Configured an automated monitoring schedule comparing live data to baseline
4. Simulated drift by sending invocations with shifted feature values
5. Detected and analyzed constraint violations in the monitoring reports

In a real production environment, the next step would be connecting monitoring
violations to automated retraining through CloudWatch Alarms, EventBridge
rules, and SageMaker Pipelines -- the closed-loop MLOps pattern.